# **차원 축소(Dimensionality Reduction)**



---



**(코랩에서)한글 폰트 지정하는 방법**

In [ ]:
# 코랩에서 실행
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import shutil

# ── 1. 나눔 폰트 설치 ──────────────────────────────────────────
!apt-get install -y fonts-nanum > /dev/null 2>&1

# ── 2. matplotlib 캐시 디렉터리 삭제 ──────────────────────────
shutil.rmtree(mpl.get_cachedir(), ignore_errors=True)

# ── 3. 폰트 등록  ───────────────────────────────────
font_path = '/usr/share/fonts/truetype/nanum/NanumBarunGothic.ttf'
fm.fontManager.addfont(font_path)
font_name = fm.FontProperties(fname=font_path).get_name()
plt.rcParams['font.family'] = font_name
plt.rcParams['font.size'] = 12
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (6, 4)



---



## **1. 주성분 분석(PCA : Principal Component Analysis)**

- 데이터의 분산을 최대한 유지하면서 특성이 많은 데이터세트의 차원을 줄이는 방법
- 원본 데이터의 분산을 최대한 보존하는 새로운 축을 찾고(주성분), 그 축에 데이터를 투영(Projection)해서 차원을 축소함
- 이를 통해 시각화와 계산이 용이하고 분석이 쉽도록 하는 **선형 변환으로 차원 축소**


### **예제 : 주성분 분석 기본**

**1. 샘플 데이터 만들기**
- **make_blobs**: '덩어리(blob)' 형태의 가상 데이터를 만들어주는 함수
- 주로 클러스터링(Clustering, 군집화) 알고리즘이나 분류(Classification) 알고리즘을 테스트하고 시각적으로 이해하는 데 사용되는, 일종의 '연습용 데이터 세트 제조기'

In [ ]:
# 라이브러리 불러오기
from sklearn.datasets import make_blobs
import matplotlib.pyplot as plt

# 실습용 데이터셋 생성하기
x, y = make_blobs(n_features=10,
                  n_samples=1000,
                  centers=5,
                  random_state=2023,
                  cluster_std=1)
print(x.shape)
print(y.shape)

plt.scatter(x[:, 0], x[:, 1], c=y)
plt.show()

**2. 훈련데이터 표준화스케일링**

In [ ]:
# 라이브러리 불러오기
from sklearn.preprocessing import StandardScaler

# 데이터 표준화하기
scaler = StandardScaler()
scaler.fit(x)
std_data = scaler.transform(x)

print(std_data)

**3. 주성분 분석 수행하기**

In [ ]:
# 라이브러리 불러오기
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA

# PCA 객체로 주성분 10개 추출하기
pca = PCA(n_components=10)
reduced_data = pca.fit_transform(std_data)

# 주성분 데이터 확인하기
pca_df = pd.DataFrame(reduced_data)

# 고윳값: 설명된 분산(explained variance) 값 확인하기
print('✅ 총 분산: ', pca.explained_variance_)
# 기여율: 설명된 분산비율(explained variance ratio) 확인하기
print('✅ 총 분산비율: ', np.sum(pca.explained_variance_ratio_))


pca_df.head()

In [ ]:
# 주성분의 설명력과 기여율 구하기
index = np.array([f'pca{n+1}' for n in range(reduced_data.shape[1])])
result = pd.DataFrame({'고유값' : pca.explained_variance_,
                       '기여율' : pca.explained_variance_ratio_},
                     index=index)
result['누적기여율']  = result['기여율'].cumsum()

# 주성분의 설명력과 기여율 확인하기
display(result)

In [ ]:
# PCA 객체로 주성분 4개 추출하기
pca = PCA(n_components=4)
X_reduced = pca.fit_transform(std_data)
print(pca.explained_variance_ratio_)

In [ ]:
# 지정한 비율에 도달할 때까지 주성분을 탐색하기
pca = PCA(n_components=0.9)
reduced_data = pca.fit_transform(std_data)
print(pca.explained_variance_ratio_)


# 주성분의 설명력과 기여율 구하기
index = np.array([f'pca{n+1}' for n in range(reduced_data.shape[1])])
result = pd.DataFrame({'고유값' : pca.explained_variance_,
                       '기여율' : pca.explained_variance_ratio_},
                     index=index)
result['누적기여율']  = result['기여율'].cumsum()

# 주성분의 설명력과 기여율 확인하기
display(result)

### **[실습] 과일 사진 데이터으로 주성분 분석 실습하기**
0. 과일 사진 데이터셋을 사용하여
1. 주성분 분석(PCA)으로 피처 차원 축소하고
2. 축소된 데이터로 원본 데이터 재구성하기
3. 설명된 분산 확인하기

#### **1-1. 주성분 분석(PCA)으로 피처 차원 축소하기**


In [ ]:
# 데이터 다운로드하기
!wget https://bit.ly/fruits_300_data -O fruits_300.npy

In [ ]:
# 필요한 라이브러리 불러오기
import numpy as np
import matplotlib.pyplot as plt

# 1.fruits_300.npy 파일 로드하기
# (300, 100, 100) : 100x100 크기의 이미지 300장.
fruits = np.load('fruits_300.npy')

# 2. 머신러닝 모델에 사용하기 위해 3차원 배열을 2차원 배열로 변경
# (300, 100, 100) -> (300, 10000): 각 이미지를 10000개의 픽셀 값을 가진 하나의 샘플로 만듦
fruits_2d = fruits.reshape(-1, 100*100)


# PCA 클래스 객체 만들기
from sklearn.decomposition import PCA

pca = PCA(n_components=50) # 주성분 개수 50개
pca.fit(fruits_2d)


# PCA 클래스가 찾은 주성분
print(pca.components_.shape)

In [ ]:
# 주성분을 100x100 이미지  크기로 출력하기
import matplotlib.pyplot as plt

def draw_fruits(arr, ratio=1):
    n = len(arr)    # n은 샘플 개수입니다
    # 한 줄에 10개씩 이미지를 그립니다. 샘플 개수를 10으로 나누어 전체 행 개수를 계산합니다.
    rows = int(np.ceil(n/10))
    # 행이 1개 이면 열 개수는 샘플 개수입니다. 그렇지 않으면 10개입니다.
    cols = n if rows < 2 else 10
    fig, axs = plt.subplots(rows, cols,
                            figsize=(cols*ratio, rows*ratio), squeeze=False)
    for i in range(rows):
        for j in range(cols):
            if i*10 + j < n:    # n 개까지만 그립니다.
                axs[i, j].imshow(arr[i*10 + j], cmap='gray_r')
            axs[i, j].axis('off')
    plt.show()


draw_fruits(pca.components_.reshape(-1, 100, 100))
# 이 주성분은 원본 데이터에서 가장 분산이 큰 방향을 순서대로 나타냄

In [ ]:
# 원본 데이터의 모양
print(fruits_2d.shape)

In [ ]:
# PCA transform() 메서드를 사용해 원본 데이터의 차원을 50으로 줄여본다.
fruits_pca = pca.transform(fruits_2d) # 50개의 특성을 가진 데이터로 변환됨
print(fruits_pca.shape)

#### **1-2. 원본 데이터 재구성하기**

In [ ]:
# 50개 차원으로 축소한 fruits_pca 데이터를 전달해 10,000(100x100)개의 특성 복원하기
fruits_inverse = pca.inverse_transform(fruits_pca)
print(fruits_inverse.shape)

In [ ]:
# 복원된 데이터 시각화하기(100x100 이미지 크기로 출력하기)
fruits_reconstruct = fruits_inverse.reshape(-1, 100, 100)

for start in [0, 100, 200]:
    draw_fruits(fruits_reconstruct[start:start+100])
    print("\n")

# 결과 이미지가 일부 흐리거나 번져보이지만
# 50개 특성에서 10,000개로 늘린 것을 감안한다면
# 50개 특성이 분산을 가장 잘 보존하도록 변화된 것을 확인할 수 있다.

In [ ]:
# 원본 데이터 이미지
# for start in [0, 100, 200]:
#     draw_fruits(fruits[start:start+100])
#     print("\n")

#### **1-3.설명된 분산 확인하기**

In [ ]:
# 설명된 분산(explained variance) 값 확인하기 : 고윳값
print(pca.explained_variance_)

print('총 분산비율: ', np.sum(pca.explained_variance_ratio_))

In [ ]:
# 설명된 분산 그래프로 시각화
plt.plot(pca.explained_variance_ratio_)
plt.xlabel('주성분 개수')
plt.ylabel('설명된 분산 비율')

### **[실습] 다른 알고리즘과 함께 사용하기**
과일 사진 원본 데이터와 PCA로 축소된 데이터를 지도학습(과일 사진 분류)에 적용해 보고 어떤 차이가 있는지 확인하기
- 과일 사진 분류
- 알고리즘 : LogistricRegression

In [ ]:
# 분류 모델 지정하기
from sklearn.linear_model import LogisticRegression
# 로지스틱 회귀 모델에서 성능 가능하기 위해 교차검증 수행
from sklearn.model_selection import cross_validate

lr = LogisticRegression()

# 타겟 지정, 0:사과, 1:파인애플, 2:바나나
target = np.array([0] * 100 + [1] * 100 + [2] * 100)


scores = cross_validate(lr, fruits_2d, target)
print(np.mean(scores['test_score']))
print(np.mean(scores['fit_time']))  # 교차 검증 폴드의 훈련시간

In [ ]:
# 로지스틱 회귀 모델에서 PCA적용한 fruits_pca 데이터로 성능 교차검증 수행
scores = cross_validate(lr, fruits_pca, target)
print(np.mean(scores['test_score']))
print(np.mean(scores['fit_time']))  # 교차 검증 폴드의 훈련시간

In [ ]:
# n_components에 주성분 갯수(정수) 대신 0~1사이의 숫자로
# PCA 클래스가 지정된 비율에 도달할때까지 자동으로 주성분을 찾도록 설정한다.
pca = PCA(n_components=0.5)  # 설명된 분산의 50%에 달하는 주성분 자동으로 찾도록 설정
pca.fit(fruits_2d)

In [ ]:
# 찾은 주성분 확인하기
print(pca.n_components_)

# 결과값: 2 --> 2개의 특성만으로 원본 데이터에 있는 분산의 50%를 표현할 수 있다.

In [ ]:
# 원본 데이터 변환하기
fruits_pca = pca.transform(fruits_2d)
print(fruits_pca.shape)

In [ ]:
# 2개의 특성만 사용하고도 교차 검증의 결과가 좋은지 확인하기
scores = cross_validate(lr, fruits_pca, target)
print(np.mean(scores['test_score']))
print(np.mean(scores['fit_time']))

# 참고 : 결과 warning메시지(반복 횟수 증가) 나오지만 무시해도 좋다.

In [ ]:
# 축소된 차원 데이터(fruits_pca) 클러스터링 찾아보기
from sklearn.cluster import KMeans

km = KMeans(n_clusters=3, random_state=42)
km.fit(fruits_pca)

In [ ]:
# 클러스터별 샘플 데이터 갯수 확인
print(np.unique(km.labels_, return_counts=True))

In [ ]:
# 클러스터별 데이터 이미지로 시각화하기
for label in range(0, 3):
    draw_fruits(fruits[km.labels_ == label])
    print("\n")

In [ ]:
# 산점도로 확인하기
for label in range(0, 3):
    data = fruits_pca[km.labels_ == label]
    plt.scatter(data[:,0], data[:,1])
plt.legend(['pineapple', 'banana', 'apple'])
plt.show()

### **[실습] 과일 사진 데이터셋을 3차원 Tensorflow 임베딩 공간에서 확인하기**

과일 사진 데이터셋을 고차원의 벡터 공간(임베딩 공간)에서 데이터들이 어떻게 분포하고 서로 어떤 관계를 맺고 있는지 시각적으로 탐색하시오.

- Tensorflow Embeddng Project
    - https://projector.tensorflow.org/
    - TensorFlow Projector는 단어, 이미지, 또는 그 외의 다양한 데이터가 고차원의 벡터 공간(임베딩 공간)에서 어떻게 분포하고 서로 어떤 관계를 맺고 있는지 시각적으로 탐색하고 분석할 수 있도록 Google에서 만든 강력한 웹 기반 도구


**1. embedding project용 데이터로 변환하기**
- 코랩에서 생성된 파일--> PC로 다운로드하기
    - vector파일 : fruits_vectors.tsv
    - metadata파일 : fruits_metadata.tsv

In [ ]:
import numpy as np
import pandas as pd

# --- 1. 데이터 로드 ---
# 'fruits_300.npy' 파일에는 (300, 100, 100) 형태의 numpy 배열이 들어있습니다.
try:
    fruits = np.load('fruits_300.npy')
    print("성공적으로 'fruits_300.npy' 파일을 로드했습니다.")
    print(f"원본 데이터 형태: {fruits.shape}")
except FileNotFoundError:
    print("오류: 'fruits_300.npy' 파일을 찾을 수 없습니다. 코드와 같은 폴더에 파일을 위치시켜 주세요.")
    exit()

# --- 2. 데이터 형태 변환 ---
# TensorFlow Projector는 2차원 데이터를 입력으로 받습니다. (샘플 수, 특성 수)
# 100x100 이미지를 10000개의 픽셀 벡터로 길게 펼쳐줍니다.
# (300, 100, 100) -> (300, 10000)
fruits_2d = fruits.reshape(-1, 100 * 100)
print(f"변환된 데이터 형태: {fruits_2d.shape}")


# --- 3. 벡터(Vectors) 파일 저장 ---
# 변환된 이미지 픽셀 데이터를 tsv(Tab-Separated Values) 파일로 저장합니다.
# TensorFlow Projector는 tsv 파일을 기본으로 지원합니다.
vectors_filename = 'fruits_vectors.tsv'
pd.DataFrame(fruits_2d).to_csv(
    vectors_filename,
    sep='\t',      # 탭으로 열을 구분.
    header=False,  # 열 이름 저장 안함
    index=False    # 행 번호는 저장 안함
)
print(f"벡터 파일 '{vectors_filename}'이 성공적으로 저장되었습니다.")


# --- 4. 메타데이터(Metadata) 파일 저장 ---
# 각 이미지가 어떤 과일인지 라벨 정보를 만들어줍니다.
# 이 데이터셋은 사과 100개, 바나나 100개, 파인애플 100개 순서로 구성되어 있습니다.
labels = ['apple'] * 100 + ['banana'] * 100 + ['pineapple'] * 100
metadata_filename = 'fruits_metadata.tsv'
pd.DataFrame(labels, columns=['Label']).to_csv(
    metadata_filename,
    sep='\t',       # 탭으로 열을 구분
    index=False,    # 행 번호는 저장 안함
    header=False    # 열 이름 저장 안함
)
print(f"메타데이터 파일 '{metadata_filename}'이 성공적으로 저장되었습니다.")

**2. 사이트에 변화된 데이터 업로드하기**
- https://projector.tensorflow.org/ 사이트 접속
    
- PC에 다운로드된 파일 사이트에 업로드하기(왼쪽 uplad 버튼 클릭)
    - vector파일 : fruits_vectors.tsv
    - metadata파일 : fruits_metadata.tsv

**3. PCA 방법과 t-SNE 방법으로 데이터 확인하기**

### **[참고] SVD로 PCA 직접 구현**
과일 사진 300장(10,000차원)으로 sklearn PCA와 같은 결과가 나오는지 수식으로 직접 만들고 확인한다.

In [ ]:
# 수식으로 직접 구현
Xc = fruits_2d - fruits_2d.mean(0)
U, S, Vt = np.linalg.svd(Xc, full_matrices=False)
evr_manual = S ** 2 / (S ** 2).sum()

# 사이킷런 PCA 사용 - 정확한 SVD로 비교
pca = PCA(n_components=100, svd_solver="full").fit(fruits_2d)

print("✅ 직접 구현 설명분산비율(앞 5개):", evr_manual[:5].round(4))
print("✅ sklearn  설명분산비율(앞 5개):", pca.explained_variance_ratio_[:5].round(4))
print("✅ 일치?", np.allclose(evr_manual[:100], pca.explained_variance_ratio_, atol=1e-6))
cum = np.cumsum(evr_manual)
for t in [0.5, 0.8, 0.9, 0.95]:
    print(f"누적 설명분산 {t:.0%} 에 필요한 주성분 수: {np.searchsorted(cum, t) + 1:4d} / 10000")

- **주성분 몇 개로 사진을 복원할 수 있을까?** (재구성 = 압축 → 복원)

In [ ]:
# 주성분 몇 개로 사진을 복원할 수 있을까? (재구성 = 압축 → 복원)
fig, axes = plt.subplots(3, 5, figsize=(10, 6))
for j, k in enumerate([1, 5, 20, 50, 300]):
    rec = Xc @ Vt[:k].T @ Vt[:k] + fruits_2d.mean(0)           # 투영 후 역변환
    mse = ((rec - fruits_2d) ** 2).mean()
    for i, idx in enumerate([0, 100, 200]):
        axes[i, j].imshow(rec[idx].reshape(100, 100), cmap="gray_r"); axes[i, j].axis("off")
    axes[0, j].set_title(f"k={k}\nMSE={mse:.0f}", fontsize=9)

plt.tight_layout(); plt.show()
fig, ax = plt.subplots(1, 5, figsize=(10, 2.2))
for j in range(5):
    ax[j].imshow(Vt[j].reshape(100, 100), cmap="RdBu"); ax[j].axis("off"); ax[j].set_title(f"PC{j+1}", fontsize=9)
plt.suptitle("주성분 = '고유 과일(eigen-fruit)' 패턴"); plt.tight_layout(); plt.show()

- **PCA → K-means : 차원을 줄이면 군집이 빨라지고 잡음이 줄어든다**

In [ ]:
import time
from sklearn.metrics import adjusted_rand_score

fruit_true = np.repeat([0, 1, 2], 100)
F50 = PCA(50, random_state=42).fit_transform(fruits_2d)     # 10,000차원 → 50차원 (B에서 자세히)

rows = []
for name, Zf in [("원본 10,000차원", fruits_2d), ("PCA 50차원", F50), ("PCA 2차원", F50[:, :2])]:
    t0 = time.time()
    lab = KMeans(3, n_init=10, random_state=42).fit_predict(Zf)
    rows.append(dict(입력=name, 소요초=round(time.time() - t0, 2),
                     ARI=round(adjusted_rand_score(fruit_true, lab), 3)))

print(pd.DataFrame(rows).to_string(index=False))



---



## **2. t-SNE(t-distributed Stochastic Neighbor Embedding)**
분산 확률적 이웃 임베딩

-
- 고차원의 복잡한 데이터를 2차원 또는 3차원으로 축소하는 방법
- **비선형적인 방법의 차원 축소**(복잡한 곡선, 곡면 구조(Manifold)를 보존하면서 저차원으로 축소 -> Manifold Learning이라고도 불림)
- 복잡한 데이터의 시각화에 주로 사용 - 고차원 공간에서의 데이터들의 유사성과 저차원 공간에서의 데이터들의 유사성 계산

### **예제 : t-SNE 기본**

**1. 합성 데이터 생성하기**

In [ ]:
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs

# 실습용 데이터셋 생성하기
x, y = make_blobs(n_features=10,
                  n_samples=100,
                  centers=3,
                  random_state=42,
                  cluster_std=2)

print(x.shape)
print(y.shape)

plt.scatter(x[:, 0], x[:, 1])
plt.show()


**2. 2차원 t-SNE 시각화하기**

In [ ]:
import pandas as pd
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt


# 2차원 t-SNE 임베딩하기
tsne_np = TSNE(n_components=2,random_state=1).fit_transform(x)

# Numpy array를 DataFrame으로 변환하기
tsne_df = pd.DataFrame(tsne_np, columns = ['component 0', 'component 1'])
tsne_df


# class target 정보 불러오기
tsne_df['target'] = y

# target별 분리하기
tsne_df_0 = tsne_df[tsne_df['target'] == 0]
tsne_df_1 = tsne_df[tsne_df['target'] == 1]
tsne_df_2 = tsne_df[tsne_df['target'] == 2]

# target별 시각화하기
plt.scatter(tsne_df_0['component 0'], tsne_df_0['component 1'],
            color = 'pink', label = 'A')
plt.scatter(tsne_df_1['component 0'], tsne_df_1['component 1'],
            color = 'purple', label = 'B')
plt.scatter(tsne_df_2['component 0'], tsne_df_2['component 1'],
            color = 'yellow', label = 'C')

plt.xlabel('component 0')
plt.ylabel('component 1')
plt.legend()
plt.show()

**3. 3차원 t-SNE 시각화하기**

In [ ]:
# 3차원 t-SNE 임베딩하기
tsne_np = TSNE(n_components=3, random_state=15).fit_transform(x)

# Numpy array를 DataFrame으로 변환하기
tsne_df = pd.DataFrame(tsne_np,
        columns = ['component 0', 'component 1', 'component 2'])

print(tsne_df)

In [ ]:
from mpl_toolkits.mplot3d import Axes3D

# 3차원 그래프 세팅하기
fig = plt.figure(figsize=(9, 6))
ax = fig.add_subplot(111, projection='3d')

# class target 정보 불러오기
tsne_df['target'] = y

# target 별 분리하기
tsne_df_0 = tsne_df[tsne_df['target'] == 0]
tsne_df_1 = tsne_df[tsne_df['target'] == 1]
tsne_df_2 = tsne_df[tsne_df['target'] == 2]

# target 별 시각화하기
ax.scatter(tsne_df_0['component 0'],
           tsne_df_0['component 1'],
           tsne_df_0['component 2'],
           color = 'pink', label = 'A')
ax.scatter(tsne_df_1['component 0'],
           tsne_df_1['component 1'],
           tsne_df_1['component 2'],
           color = 'purple', label = 'B')
ax.scatter(tsne_df_2['component 0'],
           tsne_df_2['component 1'],
           tsne_df_2['component 2'],
           color = 'yellow', label = 'C')

ax.set_xlabel('component 0')
ax.set_ylabel('component 1')
ax.set_zlabel('component 2')
ax.legend()
plt.show()

### **[실습] 과일 사진 데이터으로 t-SNE 실습하기**

In [ ]:
# 앞에서  과일 사진 데이터셋(fruits_300.np)으로 t-SNE 실습하는 코드를 추가해줘.
import pandas as pd
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt

# fruits_2d와 labels 변수는 이미 이전 셀에서 생성되었습니다.
# 2차원 t-SNE 임베딩하기
# perplexity: t-SNE의 핵심 하이퍼파라미터로, 각 데이터 포인트의 '이웃' 수를 추정하는 값입니다.
# 이 값은 지역적(local) 및 전역적(global) 데이터 구조 간의 균형을 설정합니다.
# 일반적으로 5에서 50 사이의 값을 사용하며, 데이터셋 크기에 따라 조정됩니다.
tsne_fruits_np = TSNE(n_components=2, perplexity=30, learning_rate='auto', random_state=42).fit_transform(fruits_2d)

# Numpy array를 DataFrame으로 변환하기
tsne_fruits_df = pd.DataFrame(tsne_fruits_np, columns=['component 0', 'component 1'])

# class target 정보 불러오기
tsne_fruits_df['target'] = labels

# target별 분리하기
tsne_fruits_df_apple = tsne_fruits_df[tsne_fruits_df['target'] == 'apple']
tsne_fruits_df_banana = tsne_fruits_df[tsne_fruits_df['target'] == 'banana']
tsne_fruits_df_pineapple = tsne_fruits_df[tsne_fruits_df['target'] == 'pineapple']

# target별 시각화하기
plt.figure(figsize=(8, 6))
plt.scatter(tsne_fruits_df_apple['component 0'], tsne_fruits_df_apple['component 1'],
            color='red', label='apple', alpha=0.7)
plt.scatter(tsne_fruits_df_banana['component 0'], tsne_fruits_df_banana['component 1'],
            color='yellow', label='banana', alpha=0.7)
plt.scatter(tsne_fruits_df_pineapple['component 0'], tsne_fruits_df_pineapple['component 1'],
            color='green', label='pineapple', alpha=0.7)

plt.xlabel('component 0')
plt.ylabel('component 1')
plt.title('t-SNE visualization of Fruits_300 dataset')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
tsne_model = TSNE(n_components=2, perplexity=30, learning_rate='auto', random_state=42)
tsne_model.fit(fruits_2d)

print(f"1. embedding_ (2차원 좌표):\n{tsne_model.embedding_}\n")
print(f"2. kl_divergence_ (최종 KL 발산 값): {tsne_model.kl_divergence_}\n")
print(f"3. n_iter_ (실제로 반복한 횟수): {tsne_model.n_iter_}")

# KL(Kullback-Leibler) 발산은 고차원 공간의 데이터 분포와 저차원 공간의 데이터 분포가 얼마나 유사한지를
# 측정하는 지표입니다. 이 값이 낮을수록 두 분포가 서로 비슷하다는 것을 의미한다.
# 0.1649499237537384 은 비교적 낮은 값.

### **[참고] MNIST 데이터셋으로 차원축소 알고리즘 비교**
손글씨 숫자 64차원 → 2차원 (1,797장)

In [ ]:
from sklearn.datasets import load_digits
from sklearn.manifold import TSNE, trustworthiness
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler

SEED = 42

digits = load_digits(); Xd, yd = StandardScaler().fit_transform(digits.data), digits.target
emb = {"PCA": lambda: PCA(2, random_state=SEED).fit_transform(Xd),
       "t-SNE(perp=30)": lambda: TSNE(2, perplexity=30, random_state=SEED).fit_transform(Xd),
       "t-SNE(perp=5)": lambda: TSNE(2, perplexity=5, random_state=SEED).fit_transform(Xd)}
try:
    import umap
    emb["UMAP"] = lambda: umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=SEED).fit_transform(Xd)
except ImportError:
    print("umap-learn 미설치 → UMAP 생략 (pip install umap-learn)")

fig, axes = plt.subplots(1, len(emb), figsize=(4 * len(emb), 4))
rows = []
for ax, (name, f) in zip(axes, emb.items()):
    t0 = time.time(); Y2 = f(); sec = time.time() - t0
    rows.append(dict(방법=name,
                     초=round(sec, 1),
                     신뢰성=round(trustworthiness(Xd, Y2, n_neighbors=10), 3),
                     kNN정확도_2D=round(cross_val_score(KNeighborsClassifier(10), Y2, yd, cv=5).mean(), 3)))
    ax.scatter(Y2[:, 0], Y2[:, 1], c=yd, cmap="tab10", s=4); ax.set_title(name); ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()
print(pd.DataFrame(rows).to_string(index=False))
print("주의: t-SNE/UMAP 그림에서 '군집 사이 거리·군집 크기'는 의미가 약하다 — 이웃 관계만 믿자 (Wattenberg 외, Distill 2016)")



---



## **3. 텍스트 분류**

### **[실습] 임베딩 + 차원축소 + 군집** = 실무 표준 조합 (BERTopic 축소판, Grootendorst 2022)
문서 → 임베딩 → UMAP(5차원) → HDBSCAN → 군집별 대표 단어(c-TF-IDF) → (선택) LLM이 이름 짓기

In [ ]:
!wget -q https://raw.githubusercontent.com/e9t/nsmc/master/ratings_test.txt -O ratings_test.txt

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.cluster import HDBSCAN
from sklearn.decomposition import PCA


nsmc = pd.read_csv("ratings_test.txt", sep="\t").dropna()
docs = nsmc[nsmc.document.str.len() >= 15].sample(3000, random_state=SEED).document.tolist()


def get_encoder(model_name="intfloat/multilingual-e5-small", fit_texts=None):
    try:
        from sentence_transformers import SentenceTransformer
        st = SentenceTransformer(model_name, device="cpu")
        return (lambda t: st.encode(["query: " + x for x in t], batch_size=64, normalize_embeddings=True,
                                    show_progress_bar=False)), model_name
    except Exception as e:
        print("⚠ 임베딩 모델 로드 실패 → LSA 대체 |", type(e).__name__)
        tf = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4), min_df=2, sublinear_tf=True)
        svd = TruncatedSVD(128, random_state=SEED).fit(tf.fit_transform(fit_texts))
        return (lambda t: (lambda Z: Z / (np.linalg.norm(Z, axis=1, keepdims=True) + 1e-12))(svd.transform(tf.transform(t)))), "LSA"


encode, ENC = get_encoder(fit_texts=docs)
E = encode(docs)

In [ ]:

try:
    import umap
    E5 = umap.UMAP(n_components=5, n_neighbors=15, min_dist=0.0, metric="cosine", random_state=SEED).fit_transform(E)
except ImportError:
    E5 = PCA(5, random_state=SEED).fit_transform(E)

topic = HDBSCAN(min_cluster_size=20, min_samples=10).fit_predict(E5)
print(f"인코더 {ENC} | 발견된 토픽 {topic.max() + 1}개 | 잡음(-1) {np.mean(topic == -1):.1%}")


# c-TF-IDF: 토픽별 문서를 하나로 이어 붙여 '토픽을 대표하는 단어'를 뽑는다
cv = CountVectorizer(token_pattern=r"[가-힣A-Za-z]{2,}", min_df=3)
joined = pd.Series(docs).groupby(topic).apply(" ".join)
joined = joined[joined.index >= 0]
Ct = cv.fit_transform(joined).toarray().astype(float)
tf = Ct / Ct.sum(1, keepdims=True)
idf = np.log(1 + Ct.sum(1).mean() / (Ct.sum(0) + 1))
ctfidf = tf * idf
terms = cv.get_feature_names_out()
topic_table = pd.DataFrame({"토픽": joined.index, "문서수": pd.Series(topic).value_counts().reindex(joined.index).values,
                            "대표단어": [", ".join(terms[np.argsort(-r)[:6]]) for r in ctfidf]})
print(topic_table.to_string(index=False))

#### 3-1. LLM에게 토픽 이름 짓게하기

In [ ]:
# (선택) LLM에게 토픽 이름 짓기 — GEMINI_API_KEY 필요 (04차시와 같은 방식)
def gemini_key():
    try:
        from google.colab import userdata
        return userdata.get("GEMINI_API_KEY")
    except Exception:
        return os.environ.get("GEMINI_API_KEY")

if gemini_key():
    from google import genai
    g = genai.Client(api_key=gemini_key())
    for t in topic_table.토픽[:6]:
        sample = [d for d, z in zip(docs, topic) if z == t][:8]
        name = g.interactions.create(model="gemini-3-flash-preview",
                                     input=f"다음 영화 리뷰 묶음의 공통 주제를 10자 이내 이름 하나로만: {sample}").output_text.strip()
        print(f"토픽 {t}: {name}")
else:
    print("GEMINI_API_KEY 없음 → LLM 이름 짓기 생략 (대표단어로 사람이 이름 붙이기)")